[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/benchmarks/sampling_robustness.ipynb)

# MISDA — robustness to clean sampling

This notebook repeats the controlled clean benchmark over independent sample seeds while keeping observation noise at `sigma=0`. It therefore isolates finite-sample variability from observation-noise robustness. Cases 1–11 are regular controlled diagnostics; Cases 12 and 13 remain documented adversarial limitations and are reported separately rather than treated as expected successes.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), None)
target = f"{repo_root}[benchmarks]" if repo_root is not None else "misda[benchmarks] @ git+https://github.com/monacofj/misda.git@main"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import pandas as pd
import misda
import misda.benchmarks as bench

N = 300
MISDA_SEED = 123
REPLICATE_SEEDS = tuple(range(1001, 1021))
PROBLEM_IDS = (
    "independence",
    "total_redundancy",
    "blocks_4x5",
    "blocks_2x10",
    "mixed_independent_and_blocks",
    "monotonic_redundancy",
    "antagonistic_linear_groups",
    "tradeoff_redundancies",
    "nonlinear_blocks_4x5",
    "antagonistic_nonlinear_groups",
    "overlapping_factors",
    "transitive_chain",
    "regime_switching",
)


## Sampling protocol

Each replicate generates a new clean sample from the same theoretical problem. `sigma` is fixed at zero, so `Y=Z` in every run. MISDA's own seed is held fixed at `MISDA_SEED`; only the sampled dataset changes. Recovery proportions below are finite-replicate summaries, not fitted probabilities or methodological thresholds.

In [ ]:
sampling_records = []

for problem_id in PROBLEM_IDS:
    problem = bench.PROBLEM_BY_ID[problem_id]
    for sample_seed in REPLICATE_SEEDS:
        dataset = problem.generate(N=N, seed=sample_seed, sigma=0.0)
        truth = bench.diagnostic_truth(problem, dataset.Z)
        mis_set = misda.discover(dataset.Y, name=truth["name"], seed=MISDA_SEED)
        misda.evaluate(mis_set, metrics=("pareto",), candidates=1)
        benchmark_result = misda.benchmark(mis_set, truth)
        support_reasons = {
            reason
            for candidate_support in mis_set.support.results
            for reason in candidate_support.reasons
        }
        sampling_records.append({
            "problem_id": problem_id,
            "sample_seed": sample_seed,
            "latent_exact": bool(benchmark_result.latent_exact),
            "structural_exact": bool(benchmark_result.structural_dimension_exact),
            "selected_unit_adequacy": benchmark_result.assessment["selected_unit_adequacy"],
            "support_status": mis_set.support.status,
            "supported": mis_set.support.status == "SUPPORTED",
            "transitive_chaining": "TRANSITIVE_CHAINING" in support_reasons,
            "hidden_spectral_structure": "HIDDEN_SPECTRAL_STRUCTURE" in support_reasons,
        })

sampling = pd.DataFrame.from_records(sampling_records)


In [ ]:
def _mean_if_declared(series):
    values = series.dropna()
    return float(values.astype(float).mean()) if len(values) else float("nan")

sampling_summary = (
    sampling.groupby("problem_id", sort=False)
    .agg(
        replicates=("sample_seed", "size"),
        latent_recovery=("latent_exact", "mean"),
        structural_recovery=("structural_exact", "mean"),
        selected_unit_recovery=("selected_unit_adequacy", _mean_if_declared),
        supported_rate=("supported", "mean"),
        transitive_chaining_rate=("transitive_chaining", "mean"),
        hidden_spectral_structure_rate=("hidden_spectral_structure", "mean"),
    )
    .reset_index()
)

sampling_summary


## Interpretation

For Cases 1–11, `latent_recovery` and `structural_recovery` summarize stability of the declared clean dimensions across independent samples. Cases 12 and 13 are not expected to achieve exact recovery; their diagnostic rates show whether the known `TRANSITIVE_CHAINING` and `HIDDEN_SPECTRAL_STRUCTURE` limitations remain consistently signaled across clean samples.